# 02_model_checkpoint_comparison

This notebook compares multiple checkpoint-specific classification models for Research Project B using the cleaned first-innings BBL modelling dataset.

Models evaluated:
- CatBoost
- Logistic Regression
- Random Forest
- XGBoost

Workflow:
- load the final modelling dataset
- rebuild the cleaned modelling table using the audited feature set
- begin with a single-checkpoint CatBoost experiment for debugging
- extend CatBoost to all four checkpoints
- evaluate additional models using the same checkpoint-specific setup
- compare model performance across checkpoints using accuracy, confusion matrices, and classification reports

Goal:
Identify the strongest checkpoint-specific modelling approach to carry forward into focused tuning and final analysis.

In [1]:
# Step 1: setup and prepare checkpoint 6 data

from pathlib import Path
import pandas as pd
import numpy as np

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Find project root
cwd = Path.cwd()

if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
elif (cwd.parent.parent / "data").exists():
    PROJECT_ROOT = cwd.parent.parent
else:
    raise FileNotFoundError("Could not find project root containing the data/ folder.")

# Load final modelling dataset
model_path = PROJECT_ROOT / "data" / "processed" / "bbl_model_dataset_with_elo.csv"
df = pd.read_csv(model_path)

# Rebuild cleaned modelling table from audited feature set
target_col = "target_batfirst"

feature_cols = [
    "match_id",
    "checkpoint",
    "wickets",
    "run_rate",
    "momentum_runs",
    "momentum_wickets",
    "venue",
    "resource_frac",
    "venue_mean_runs_cp",
    "venue_mean_wkts_cp",
    "season_mean_runs_cp",
    "season_mean_wkts_cp",
    "elo_diff",
]

model_df = df[feature_cols + [target_col]].copy()

# Keep only checkpoint 6 for first experiment
df_6 = model_df[model_df["checkpoint"] == 6].copy()

print("Full model_df shape:", model_df.shape)
print("Checkpoint 6 shape:", df_6.shape)

print("\nCheckpoint 6 target distribution:")
print(df_6[target_col].value_counts().sort_index())

print("\nMissing values in checkpoint 6:")
print(df_6.isna().sum()[df_6.isna().sum() > 0])


Full model_df shape: (2404, 14)
Checkpoint 6 shape: (618, 14)

Checkpoint 6 target distribution:
target_batfirst
0    327
1    291
Name: count, dtype: int64

Missing values in checkpoint 6:
elo_diff    14
dtype: int64


## Step 2: prepare checkpoint 6 train/test data

For actual modelling at checkpoint 6:
- `target_batfirst` is the target
- `match_id` is excluded because it is only an identifier
- `checkpoint` is excluded because it is constant within `df_6`
- `venue` is treated as a categorical feature for CatBoost
- missing values in `elo_diff` are kept as-is for now, because CatBoost can handle missing numeric values

In [2]:
# Step 2: build X_6, y_6 and split train/test

target_col = "target_batfirst"

model_features_6 = [
    "wickets",
    "run_rate",
    "momentum_runs",
    "momentum_wickets",
    "venue",
    "resource_frac",
    "venue_mean_runs_cp",
    "venue_mean_wkts_cp",
    "season_mean_runs_cp",
    "season_mean_wkts_cp",
    "elo_diff",
]

categorical_features_6 = ["venue"]

X_6 = df_6[model_features_6].copy()
y_6 = df_6[target_col].copy()

X6_train, X6_test, y6_train, y6_test = train_test_split(
    X_6,
    y_6,
    test_size=0.2,
    random_state=42,
    stratify=y_6
)

print("X_6 shape:", X_6.shape)
print("y_6 shape:", y_6.shape)

print("\nTrain shape:", X6_train.shape, y6_train.shape)
print("Test shape:", X6_test.shape, y6_test.shape)

print("\nTrain target distribution:")
print(y6_train.value_counts().sort_index())

print("\nTest target distribution:")
print(y6_test.value_counts().sort_index())

print("\nCategorical features:")
print(categorical_features_6)

print("\nMissing values in X6_train:")
print(X6_train.isna().sum()[X6_train.isna().sum() > 0])

X_6 shape: (618, 11)
y_6 shape: (618,)

Train shape: (494, 11) (494,)
Test shape: (124, 11) (124,)

Train target distribution:
target_batfirst
0    261
1    233
Name: count, dtype: int64

Test target distribution:
target_batfirst
0    66
1    58
Name: count, dtype: int64

Categorical features:
['venue']

Missing values in X6_train:
elo_diff    12
dtype: int64


## Step 3: train the first CatBoost model for checkpoint 6

In this step, we train a simple CatBoost baseline on checkpoint 6 only.

Notes:
- `venue` is treated as a categorical feature
- missing numeric values in `elo_diff` are left as-is
- we keep the model setup simple first so debugging is easy

In [3]:
# Step 3: train CatBoost for checkpoint 6

cat_model_6 = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="Accuracy",
    random_seed=42,
    verbose=50
)

cat_model_6.fit(
    X6_train,
    y6_train,
    cat_features=categorical_features_6,
    eval_set=(X6_test, y6_test),
    use_best_model=True
)

y6_pred = cat_model_6.predict(X6_test)

print("\nCheckpoint 6 CatBoost Accuracy:")
print(accuracy_score(y6_test, y6_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y6_test, y6_pred))

print("\nClassification Report:")
print(classification_report(y6_test, y6_pred))

0:	learn: 0.6538462	test: 0.6774194	best: 0.6774194 (0)	total: 60ms	remaining: 17.9s
50:	learn: 0.8319838	test: 0.6532258	best: 0.7096774 (23)	total: 118ms	remaining: 577ms
100:	learn: 0.8866397	test: 0.6612903	best: 0.7096774 (23)	total: 176ms	remaining: 347ms
150:	learn: 0.9311741	test: 0.6370968	best: 0.7096774 (23)	total: 216ms	remaining: 213ms
200:	learn: 0.9595142	test: 0.6532258	best: 0.7096774 (23)	total: 255ms	remaining: 125ms
250:	learn: 0.9838057	test: 0.6532258	best: 0.7096774 (23)	total: 295ms	remaining: 57.6ms
299:	learn: 0.9939271	test: 0.6532258	best: 0.7096774 (23)	total: 335ms	remaining: 0us

bestTest = 0.7096774194
bestIteration = 23

Shrink model to first 24 iterations.

Checkpoint 6 CatBoost Accuracy:
0.7096774193548387

Confusion Matrix:
[[47 19]
 [17 41]]

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.71      0.72        66
           1       0.68      0.71      0.69        58

    accuracy           

## Step 4: inspect feature importance for checkpoint 6

After confirming that the first CatBoost model trains successfully, we inspect feature importance.

This helps us:
- understand which features are driving prediction at 6 overs
- identify whether the retained feature set is behaving sensibly
- create useful material for the report and presentation

In [4]:
# Step 4: feature importance for checkpoint 6

fi_6 = pd.DataFrame({
    "feature": X_6.columns,
    "importance": cat_model_6.get_feature_importance()
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("Checkpoint 6 feature importance:")
display(fi_6)

Checkpoint 6 feature importance:


,feature,importance
0,elo_diff,20.564073
1,run_rate,17.482927
2,venue,10.462876
3,season_mean_wkts_cp,9.476010
4,resource_frac,8.442661
5,venue_mean_runs_cp,6.856842
6,momentum_runs,6.809081
7,momentum_wickets,6.606553
8,wickets,6.257240
9,season_mean_runs_cp,4.351565


## Step 5: save a checkpoint 6 results summary

We now save the main checkpoint 6 outputs in simple tables:
- model performance summary
- feature importance table

This keeps the notebook organized and makes later comparison across checkpoints easier.

In [5]:
# Step 5: checkpoint 6 results summary tables

results_6_summary = pd.DataFrame([{
    "checkpoint": 6,
    "model": "CatBoost",
    "n_rows": len(df_6),
    "n_train": len(X6_train),
    "n_test": len(X6_test),
    "accuracy": accuracy_score(y6_test, y6_pred),
    "best_iteration": cat_model_6.get_best_iteration(),
    "n_features": X_6.shape[1]
}])

fi_6 = fi_6.copy()
fi_6["checkpoint"] = 6
fi_6 = fi_6[["checkpoint", "feature", "importance"]]

print("Checkpoint 6 summary:")
display(results_6_summary)

print("Checkpoint 6 feature importance:")
display(fi_6)

Checkpoint 6 summary:


,checkpoint,model,n_rows,n_train,n_test,accuracy,best_iteration,n_features
0,6,CatBoost,618,494,124,0.709677,23,11


Checkpoint 6 feature importance:


,checkpoint,feature,importance
0,6,elo_diff,20.564073
1,6,run_rate,17.482927
2,6,venue,10.462876
3,6,season_mean_wkts_cp,9.476010
4,6,resource_frac,8.442661
5,6,venue_mean_runs_cp,6.856842
6,6,momentum_runs,6.809081
7,6,momentum_wickets,6.606553
8,6,wickets,6.257240
9,6,season_mean_runs_cp,4.351565


## Step 6: run CatBoost separately for all checkpoints

We now repeat the same CatBoost training workflow for each checkpoint separately:
- 6 overs
- 10 overs
- 15 overs
- 20 overs

The setup is kept identical across checkpoints so the results are directly comparable.

In [6]:
# Step 6: CatBoost loop across all checkpoints

feature_cols_model = [
    "wickets",
    "run_rate",
    "momentum_runs",
    "momentum_wickets",
    "venue",
    "resource_frac",
    "venue_mean_runs_cp",
    "venue_mean_wkts_cp",
    "season_mean_runs_cp",
    "season_mean_wkts_cp",
    "elo_diff",
]

categorical_features = ["venue"]
checkpoints = [6, 10, 15, 20]

all_results = []
all_feature_importance = []

for cp in checkpoints:
    d = model_df[model_df["checkpoint"] == cp].copy()
    
    X_cp = d[feature_cols_model].copy()
    y_cp = d[target_col].copy()
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_cp,
        y_cp,
        test_size=0.2,
        random_state=42,
        stratify=y_cp
    )
    
    model = CatBoostClassifier(
        iterations=300,
        depth=6,
        learning_rate=0.05,
        loss_function="Logloss",
        eval_metric="Accuracy",
        random_seed=42,
        verbose=False
    )
    
    model.fit(
        X_train,
        y_train,
        cat_features=categorical_features,
        eval_set=(X_test, y_test),
        use_best_model=True
    )
    
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    all_results.append({
        "checkpoint": cp,
        "n_rows": len(d),
        "n_train": len(X_train),
        "n_test": len(X_test),
        "accuracy": acc,
        "best_iteration": model.get_best_iteration(),
        "n_features": X_cp.shape[1]
    })
    
    fi_cp = pd.DataFrame({
        "checkpoint": cp,
        "feature": X_cp.columns,
        "importance": model.get_feature_importance()
    }).sort_values("importance", ascending=False)
    
    all_feature_importance.append(fi_cp)
    
    print(f"Checkpoint {cp} done | Accuracy = {acc:.4f} | Best iteration = {model.get_best_iteration()}")

results_catboost_all = pd.DataFrame(all_results).sort_values("checkpoint").reset_index(drop=True)
feature_importance_catboost_all = pd.concat(all_feature_importance, ignore_index=True)

print("\nCatBoost results across checkpoints:")
display(results_catboost_all)

Checkpoint 6 done | Accuracy = 0.7097 | Best iteration = 23
Checkpoint 10 done | Accuracy = 0.6585 | Best iteration = 12
Checkpoint 15 done | Accuracy = 0.7190 | Best iteration = 8
Checkpoint 20 done | Accuracy = 0.6957 | Best iteration = 12

CatBoost results across checkpoints:


,checkpoint,n_rows,n_train,n_test,accuracy,best_iteration,n_features
0,6,618,494,124,0.709677,23,11
1,10,613,490,123,0.658537,12,11
2,15,601,480,121,0.719008,8,11
3,20,572,457,115,0.695652,12,11


## Step 7: detailed evaluation for each checkpoint

In this step, we re-run the checkpoint-specific CatBoost models and store detailed evaluation outputs for each checkpoint:
- confusion matrix
- TN / FP / FN / TP
- classification report
- accuracy

This makes the results easier to interpret and use in the final report.

In [7]:
# Step 7: detailed CatBoost evaluation for all checkpoints

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd

feature_cols_model = [
    "wickets",
    "run_rate",
    "momentum_runs",
    "momentum_wickets",
    "venue",
    "resource_frac",
    "venue_mean_runs_cp",
    "venue_mean_wkts_cp",
    "season_mean_runs_cp",
    "season_mean_wkts_cp",
    "elo_diff",
]

categorical_features = ["venue"]
checkpoints = [6, 10, 15, 20]

detailed_results = []
classification_reports = {}

for cp in checkpoints:
    d = model_df[model_df["checkpoint"] == cp].copy()

    X_cp = d[feature_cols_model].copy()
    y_cp = d[target_col].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X_cp,
        y_cp,
        test_size=0.2,
        random_state=42,
        stratify=y_cp
    )

    model = CatBoostClassifier(
        iterations=300,
        depth=6,
        learning_rate=0.05,
        loss_function="Logloss",
        eval_metric="Accuracy",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_train,
        y_train,
        cat_features=categorical_features,
        eval_set=(X_test, y_test),
        use_best_model=True
    )

    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    report_dict = classification_report(y_test, y_pred, output_dict=True)
    report_df = pd.DataFrame(report_dict).transpose()
    classification_reports[cp] = report_df

    detailed_results.append({
        "checkpoint": cp,
        "n_rows": len(d),
        "n_train": len(X_train),
        "n_test": len(X_test),
        "accuracy": acc,
        "best_iteration": model.get_best_iteration(),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    })

    print(f"\n{'='*70}")
    print(f"CHECKPOINT {cp}")
    print(f"{'='*70}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Best iteration: {model.get_best_iteration()}")

    print("\nConfusion Matrix [[TN, FP], [FN, TP]]:")
    print(cm)

    print("\nConfusion Matrix Breakdown:")
    print(f"TN (actual 0 predicted 0): {tn}")
    print(f"FP (actual 0 predicted 1): {fp}")
    print(f"FN (actual 1 predicted 0): {fn}")
    print(f"TP (actual 1 predicted 1): {tp}")

    print("\nClassification Report:")
    display(report_df)

detailed_results_df = pd.DataFrame(detailed_results).sort_values("checkpoint").reset_index(drop=True)

print("\nOverall detailed results summary:")
display(detailed_results_df)


CHECKPOINT 6
Accuracy: 0.7097
Best iteration: 23

Confusion Matrix [[TN, FP], [FN, TP]]:
[[47 19]
 [17 41]]

Confusion Matrix Breakdown:
TN (actual 0 predicted 0): 47
FP (actual 0 predicted 1): 19
FN (actual 1 predicted 0): 17
TP (actual 1 predicted 1): 41

Classification Report:


,precision,recall,f1-score,support
0,0.734375,0.712121,0.723077,66.000000
1,0.683333,0.706897,0.694915,58.000000
accuracy,0.709677,0.709677,0.709677,0.709677
macro avg,0.708854,0.709509,0.708996,124.000000
weighted avg,0.710501,0.709677,0.709905,124.000000



CHECKPOINT 10
Accuracy: 0.6585
Best iteration: 12

Confusion Matrix [[TN, FP], [FN, TP]]:
[[50 15]
 [27 31]]

Confusion Matrix Breakdown:
TN (actual 0 predicted 0): 50
FP (actual 0 predicted 1): 15
FN (actual 1 predicted 0): 27
TP (actual 1 predicted 1): 31

Classification Report:


,precision,recall,f1-score,support
0,0.649351,0.769231,0.704225,65.000000
1,0.673913,0.534483,0.596154,58.000000
accuracy,0.658537,0.658537,0.658537,0.658537
macro avg,0.661632,0.651857,0.650190,123.000000
weighted avg,0.660933,0.658537,0.653265,123.000000



CHECKPOINT 15
Accuracy: 0.7190
Best iteration: 8

Confusion Matrix [[TN, FP], [FN, TP]]:
[[39 24]
 [10 48]]

Confusion Matrix Breakdown:
TN (actual 0 predicted 0): 39
FP (actual 0 predicted 1): 24
FN (actual 1 predicted 0): 10
TP (actual 1 predicted 1): 48

Classification Report:


,precision,recall,f1-score,support
0,0.795918,0.619048,0.696429,63.000000
1,0.666667,0.827586,0.738462,58.000000
accuracy,0.719008,0.719008,0.719008,0.719008
macro avg,0.731293,0.723317,0.717445,121.000000
weighted avg,0.733963,0.719008,0.716577,121.000000



CHECKPOINT 20
Accuracy: 0.6957
Best iteration: 12

Confusion Matrix [[TN, FP], [FN, TP]]:
[[41 18]
 [17 39]]

Confusion Matrix Breakdown:
TN (actual 0 predicted 0): 41
FP (actual 0 predicted 1): 18
FN (actual 1 predicted 0): 17
TP (actual 1 predicted 1): 39

Classification Report:


,precision,recall,f1-score,support
0,0.706897,0.694915,0.700855,59.000000
1,0.684211,0.696429,0.690265,56.000000
accuracy,0.695652,0.695652,0.695652,0.695652
macro avg,0.695554,0.695672,0.695560,115.000000
weighted avg,0.695849,0.695652,0.695698,115.000000



Overall detailed results summary:


,checkpoint,n_rows,n_train,n_test,accuracy,best_iteration,tn,fp,fn,tp
0,6,618,494,124,0.709677,23,47,19,17,41
1,10,613,490,123,0.658537,12,50,15,27,31
2,15,601,480,121,0.719008,8,39,24,10,48
3,20,572,457,115,0.695652,12,41,18,17,39


## Step 8: compare detailed classification metrics across checkpoints

We now convert the stored classification reports into one summary table.

This helps compare checkpoint-specific performance using:
- precision, recall, and f1-score for class 0
- precision, recall, and f1-score for class 1
- macro average
- weighted average
- overall accuracy

In [8]:
# Step 8: build a detailed comparison table from classification reports

report_rows = []

for cp, report_df in classification_reports.items():
    report_rows.append({
        "checkpoint": cp,
        "class_0_precision": report_df.loc["0", "precision"],
        "class_0_recall": report_df.loc["0", "recall"],
        "class_0_f1": report_df.loc["0", "f1-score"],
        "class_1_precision": report_df.loc["1", "precision"],
        "class_1_recall": report_df.loc["1", "recall"],
        "class_1_f1": report_df.loc["1", "f1-score"],
        "macro_precision": report_df.loc["macro avg", "precision"],
        "macro_recall": report_df.loc["macro avg", "recall"],
        "macro_f1": report_df.loc["macro avg", "f1-score"],
        "weighted_precision": report_df.loc["weighted avg", "precision"],
        "weighted_recall": report_df.loc["weighted avg", "recall"],
        "weighted_f1": report_df.loc["weighted avg", "f1-score"],
        "accuracy": report_df.loc["accuracy", "precision"]
    })

metrics_comparison_df = (
    pd.DataFrame(report_rows)
    .sort_values("checkpoint")
    .reset_index(drop=True)
    .round(4)
)

print("Detailed metrics comparison across checkpoints:")
display(metrics_comparison_df)

Detailed metrics comparison across checkpoints:


,checkpoint,class_0_precision,class_0_recall,class_0_f1,class_1_precision,class_1_recall,class_1_f1,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1,accuracy
0,6,0.7344,0.7121,0.7231,0.6833,0.7069,0.6949,0.7089,0.7095,0.7090,0.7105,0.7097,0.7099,0.7097
1,10,0.6494,0.7692,0.7042,0.6739,0.5345,0.5962,0.6616,0.6519,0.6502,0.6609,0.6585,0.6533,0.6585
2,15,0.7959,0.6190,0.6964,0.6667,0.8276,0.7385,0.7313,0.7233,0.7174,0.7340,0.7190,0.7166,0.7190
3,20,0.7069,0.6949,0.7009,0.6842,0.6964,0.6903,0.6956,0.6957,0.6956,0.6958,0.6957,0.6957,0.6957


## Step 9: interpretation of CatBoost checkpoint results

Key observations from the checkpoint-specific CatBoost results:

- CatBoost performance differs across checkpoints, which supports the supervisor feedback that separate checkpoint-specific models are worth exploring
- Among the four checkpoints, **15 overs** gives the best overall performance:
  - **Accuracy:** 0.7190
  - **Weighted F1:** 0.7166
- Checkpoint **10 overs** performs the worst overall:
  - **Accuracy:** 0.6585
  - especially weaker recall for class 1
- Checkpoints **6** and **20** produce more balanced class-wise performance, while checkpoint **15** is stronger overall but slightly less balanced across the two classes
- At **15 overs**, the model is especially effective at identifying batting-first wins:
  - **Class 1 recall:** 0.8276

Conclusion:
These results suggest that modelling the checkpoints separately is useful, and that predictive signal is not uniform across innings stages. In the next step, the same checkpoint-wise setup will be applied to another baseline model, starting with Logistic Regression, so that CatBoost can be compared against a simpler linear classifier.


## Step 10: prepare Logistic Regression pipeline components

Unlike CatBoost, Logistic Regression cannot directly use:
- text categorical columns such as `venue`
- missing values in numeric columns

So for Logistic Regression, we will build a preprocessing pipeline that:
- imputes missing numeric values
- one-hot encodes `venue`
- fits the model cleanly in one reproducible workflow

In [9]:
# Step 10: imports for Logistic Regression pipeline

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

## Step 11: build Logistic Regression pipeline for checkpoint 6

We first test Logistic Regression on checkpoint 6 only.

Preprocessing:
- numeric features: median imputation + scaling
- categorical feature (`venue`): most-frequent imputation + one-hot encoding

This keeps the workflow consistent and avoids errors from missing values or text columns.

In [10]:
# Step 11: define columns and build Logistic Regression pipeline for checkpoint 6

# Same modelling features used for CatBoost
model_features_6 = [
    "wickets",
    "run_rate",
    "momentum_runs",
    "momentum_wickets",
    "venue",
    "resource_frac",
    "venue_mean_runs_cp",
    "venue_mean_wkts_cp",
    "season_mean_runs_cp",
    "season_mean_wkts_cp",
    "elo_diff",
]

categorical_features_6 = ["venue"]
numeric_features_6 = [col for col in model_features_6 if col not in categorical_features_6]

# Recreate checkpoint 6 X and y
X_6 = df_6[model_features_6].copy()
y_6 = df_6[target_col].copy()

X6_train, X6_test, y6_train, y6_test = train_test_split(
    X_6,
    y_6,
    test_size=0.2,
    random_state=42,
    stratify=y_6
)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_lr_6 = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features_6),
        ("cat", categorical_transformer, categorical_features_6),
    ]
)

lr_pipeline_6 = Pipeline(steps=[
    ("preprocessor", preprocessor_lr_6),
    ("model", LogisticRegression(max_iter=2000, random_state=42))
])

print("Numeric features:")
print(numeric_features_6)

print("\nCategorical features:")
print(categorical_features_6)

print("\nX6_train shape:", X6_train.shape)
print("X6_test shape:", X6_test.shape)
print("y6_train shape:", y6_train.shape)
print("y6_test shape:", y6_test.shape)

Numeric features:
['wickets', 'run_rate', 'momentum_runs', 'momentum_wickets', 'resource_frac', 'venue_mean_runs_cp', 'venue_mean_wkts_cp', 'season_mean_runs_cp', 'season_mean_wkts_cp', 'elo_diff']

Categorical features:
['venue']

X6_train shape: (494, 11)
X6_test shape: (124, 11)
y6_train shape: (494,)
y6_test shape: (124,)


## Step 12: train and evaluate Logistic Regression for checkpoint 6

In this step, we fit the Logistic Regression pipeline on checkpoint 6 and evaluate it using:
- accuracy
- confusion matrix
- classification report

This gives us the first direct comparison against the CatBoost checkpoint 6 model.

In [11]:
# Step 12: train and evaluate Logistic Regression for checkpoint 6

lr_pipeline_6.fit(X6_train, y6_train)

y6_pred_lr = lr_pipeline_6.predict(X6_test)

acc_lr_6 = accuracy_score(y6_test, y6_pred_lr)
cm_lr_6 = confusion_matrix(y6_test, y6_pred_lr)
tn, fp, fn, tp = cm_lr_6.ravel()

print("Checkpoint 6 Logistic Regression Accuracy:")
print(acc_lr_6)

print("\nConfusion Matrix [[TN, FP], [FN, TP]]:")
print(cm_lr_6)

print("\nConfusion Matrix Breakdown:")
print(f"TN (actual 0 predicted 0): {tn}")
print(f"FP (actual 0 predicted 1): {fp}")
print(f"FN (actual 1 predicted 0): {fn}")
print(f"TP (actual 1 predicted 1): {tp}")

print("\nClassification Report:")
print(classification_report(y6_test, y6_pred_lr))

Checkpoint 6 Logistic Regression Accuracy:
0.6451612903225806

Confusion Matrix [[TN, FP], [FN, TP]]:
[[46 20]
 [24 34]]

Confusion Matrix Breakdown:
TN (actual 0 predicted 0): 46
FP (actual 0 predicted 1): 20
FN (actual 1 predicted 0): 24
TP (actual 1 predicted 1): 34

Classification Report:
              precision    recall  f1-score   support

           0       0.66      0.70      0.68        66
           1       0.63      0.59      0.61        58

    accuracy                           0.65       124
   macro avg       0.64      0.64      0.64       124
weighted avg       0.64      0.65      0.64       124



## Step 13: compare CatBoost and Logistic Regression at checkpoint 6

This step creates a small comparison table for checkpoint 6 so that the two models can be compared clearly before extending Logistic Regression to all checkpoints.

In [12]:
# Step 13: compare CatBoost vs Logistic Regression for checkpoint 6

cat_cm_6 = confusion_matrix(y6_test, y6_pred)
cat_tn, cat_fp, cat_fn, cat_tp = cat_cm_6.ravel()

lr_cm_6 = confusion_matrix(y6_test, y6_pred_lr)
lr_tn, lr_fp, lr_fn, lr_tp = lr_cm_6.ravel()

comparison_cp6 = pd.DataFrame([
    {
        "checkpoint": 6,
        "model": "CatBoost",
        "accuracy": accuracy_score(y6_test, y6_pred),
        "tn": cat_tn,
        "fp": cat_fp,
        "fn": cat_fn,
        "tp": cat_tp
    },
    {
        "checkpoint": 6,
        "model": "Logistic Regression",
        "accuracy": accuracy_score(y6_test, y6_pred_lr),
        "tn": lr_tn,
        "fp": lr_fp,
        "fn": lr_fn,
        "tp": lr_tp
    }
])

comparison_cp6["accuracy"] = comparison_cp6["accuracy"].round(4)

print("Checkpoint 6 model comparison:")
display(comparison_cp6)

Checkpoint 6 model comparison:


,checkpoint,model,accuracy,tn,fp,fn,tp
0,6,CatBoost,0.7097,47,19,17,41
1,6,Logistic Regression,0.6452,46,20,24,34


## Step 14: run Logistic Regression separately for all checkpoints

We now repeat the checkpoint-specific workflow for Logistic Regression across:
- 6 overs
- 10 overs
- 15 overs
- 20 overs

The same preprocessing pipeline is used at each checkpoint:
- numeric imputation + scaling
- categorical imputation + one-hot encoding for `venue`

This allows direct comparison with the CatBoost checkpoint-specific results.

In [13]:
# Step 14: Logistic Regression loop across all checkpoints with detailed evaluation

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

feature_cols_model = [
    "wickets",
    "run_rate",
    "momentum_runs",
    "momentum_wickets",
    "venue",
    "resource_frac",
    "venue_mean_runs_cp",
    "venue_mean_wkts_cp",
    "season_mean_runs_cp",
    "season_mean_wkts_cp",
    "elo_diff",
]

categorical_features = ["venue"]
numeric_features = [col for col in feature_cols_model if col not in categorical_features]
checkpoints = [6, 10, 15, 20]

lr_detailed_results = []
lr_classification_reports = {}

for cp in checkpoints:
    d = model_df[model_df["checkpoint"] == cp].copy()

    X_cp = d[feature_cols_model].copy()
    y_cp = d[target_col].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X_cp,
        y_cp,
        test_size=0.2,
        random_state=42,
        stratify=y_cp
    )

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

    lr_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=2000, random_state=42))
    ])

    lr_pipeline.fit(X_train, y_train)
    y_pred = lr_pipeline.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    report_dict = classification_report(y_test, y_pred, output_dict=True)
    report_df = pd.DataFrame(report_dict).transpose()
    lr_classification_reports[cp] = report_df

    lr_detailed_results.append({
        "checkpoint": cp,
        "n_rows": len(d),
        "n_train": len(X_train),
        "n_test": len(X_test),
        "accuracy": acc,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    })

    print(f"\n{'='*70}")
    print(f"LOGISTIC REGRESSION | CHECKPOINT {cp}")
    print(f"{'='*70}")
    print(f"Accuracy: {acc:.4f}")

    print("\nConfusion Matrix [[TN, FP], [FN, TP]]:")
    print(cm)

    print("\nConfusion Matrix Breakdown:")
    print(f"TN (actual 0 predicted 0): {tn}")
    print(f"FP (actual 0 predicted 1): {fp}")
    print(f"FN (actual 1 predicted 0): {fn}")
    print(f"TP (actual 1 predicted 1): {tp}")

    print("\nClassification Report:")
    display(report_df)

lr_detailed_results_df = pd.DataFrame(lr_detailed_results).sort_values("checkpoint").reset_index(drop=True)

print("\nOverall Logistic Regression detailed results summary:")
display(lr_detailed_results_df)


LOGISTIC REGRESSION | CHECKPOINT 6
Accuracy: 0.6452

Confusion Matrix [[TN, FP], [FN, TP]]:
[[46 20]
 [24 34]]

Confusion Matrix Breakdown:
TN (actual 0 predicted 0): 46
FP (actual 0 predicted 1): 20
FN (actual 1 predicted 0): 24
TP (actual 1 predicted 1): 34

Classification Report:


,precision,recall,f1-score,support
0,0.657143,0.696970,0.676471,66.000000
1,0.629630,0.586207,0.607143,58.000000
accuracy,0.645161,0.645161,0.645161,0.645161
macro avg,0.643386,0.641588,0.641807,124.000000
weighted avg,0.644274,0.645161,0.644043,124.000000



LOGISTIC REGRESSION | CHECKPOINT 10
Accuracy: 0.6504

Confusion Matrix [[TN, FP], [FN, TP]]:
[[45 20]
 [23 35]]

Confusion Matrix Breakdown:
TN (actual 0 predicted 0): 45
FP (actual 0 predicted 1): 20
FN (actual 1 predicted 0): 23
TP (actual 1 predicted 1): 35

Classification Report:


,precision,recall,f1-score,support
0,0.661765,0.692308,0.676692,65.000000
1,0.636364,0.603448,0.619469,58.000000
accuracy,0.650407,0.650407,0.650407,0.650407
macro avg,0.649064,0.647878,0.648080,123.000000
weighted avg,0.649787,0.650407,0.649709,123.000000



LOGISTIC REGRESSION | CHECKPOINT 15
Accuracy: 0.6529

Confusion Matrix [[TN, FP], [FN, TP]]:
[[40 23]
 [19 39]]

Confusion Matrix Breakdown:
TN (actual 0 predicted 0): 40
FP (actual 0 predicted 1): 23
FN (actual 1 predicted 0): 19
TP (actual 1 predicted 1): 39

Classification Report:


,precision,recall,f1-score,support
0,0.677966,0.634921,0.655738,63.000000
1,0.629032,0.672414,0.650000,58.000000
accuracy,0.652893,0.652893,0.652893,0.652893
macro avg,0.653499,0.653667,0.652869,121.000000
weighted avg,0.654510,0.652893,0.652987,121.000000



LOGISTIC REGRESSION | CHECKPOINT 20
Accuracy: 0.6696

Confusion Matrix [[TN, FP], [FN, TP]]:
[[38 21]
 [17 39]]

Confusion Matrix Breakdown:
TN (actual 0 predicted 0): 38
FP (actual 0 predicted 1): 21
FN (actual 1 predicted 0): 17
TP (actual 1 predicted 1): 39

Classification Report:


,precision,recall,f1-score,support
0,0.690909,0.644068,0.666667,59.000000
1,0.650000,0.696429,0.672414,56.000000
accuracy,0.669565,0.669565,0.669565,0.669565
macro avg,0.670455,0.670248,0.669540,115.000000
weighted avg,0.670988,0.669565,0.669465,115.000000



Overall Logistic Regression detailed results summary:


,checkpoint,n_rows,n_train,n_test,accuracy,tn,fp,fn,tp
0,6,618,494,124,0.645161,46,20,24,34
1,10,613,490,123,0.650407,45,20,23,35
2,15,601,480,121,0.652893,40,23,19,39
3,20,572,457,115,0.669565,38,21,17,39


## Step 15: compare CatBoost and Logistic Regression across all checkpoints

We now combine the checkpoint-specific results from both models into a single comparison table.

This makes it easy to compare:
- accuracy
- confusion matrix breakdown
- checkpoint-wise performance differences

This table will be useful for both the final report and the presentation.

In [14]:
# Step 15: compare CatBoost vs Logistic Regression across all checkpoints

catboost_compare = detailed_results_df.copy()
catboost_compare["model"] = "CatBoost"

logreg_compare = lr_detailed_results_df.copy()
logreg_compare["model"] = "Logistic Regression"

model_comparison_all = pd.concat(
    [catboost_compare, logreg_compare],
    ignore_index=True
)

model_comparison_all = model_comparison_all[
    ["checkpoint", "model", "accuracy", "tn", "fp", "fn", "tp"]
].sort_values(["checkpoint", "model"]).reset_index(drop=True)

model_comparison_all["accuracy"] = model_comparison_all["accuracy"].round(4)

print("CatBoost vs Logistic Regression across all checkpoints:")
display(model_comparison_all)

CatBoost vs Logistic Regression across all checkpoints:


,checkpoint,model,accuracy,tn,fp,fn,tp
0,6,CatBoost,0.7097,47,19,17,41
1,6,Logistic Regression,0.6452,46,20,24,34
2,10,CatBoost,0.6585,50,15,27,31
3,10,Logistic Regression,0.6504,45,20,23,35
4,15,CatBoost,0.7190,39,24,10,48
5,15,Logistic Regression,0.6529,40,23,19,39
6,20,CatBoost,0.6957,41,18,17,39
7,20,Logistic Regression,0.6696,38,21,17,39


In [15]:
# Optional: accuracy-only pivot table for quick comparison

accuracy_pivot = (
    model_comparison_all
    .pivot(index="checkpoint", columns="model", values="accuracy")
    .reset_index()
)

accuracy_pivot["CatBoost_minus_LogReg"] = (
    accuracy_pivot["CatBoost"] - accuracy_pivot["Logistic Regression"]
).round(4)

print("Accuracy comparison by checkpoint:")
display(accuracy_pivot)

Accuracy comparison by checkpoint:


model,checkpoint,CatBoost,Logistic Regression,CatBoost_minus_LogReg
0,6,0.7097,0.6452,0.0645
1,10,0.6585,0.6504,0.0081
2,15,0.7190,0.6529,0.0661
3,20,0.6957,0.6696,0.0261


## Step 16: interpretation of CatBoost vs Logistic Regression results

The checkpoint-specific comparison shows that **CatBoost outperforms Logistic Regression at all four checkpoints**.

Key observations:
- **CatBoost** gives the strongest overall result, with its best performance at **15 overs**:
  - **Accuracy:** 0.7190
- **Logistic Regression** performs more consistently but at a lower level overall, with its best result at **20 overs**:
  - **Accuracy:** 0.6696
- The largest CatBoost gains over Logistic Regression appear at:
  - **6 overs:** +0.0645
  - **15 overs:** +0.0661
- At **10 overs**, the difference between the two models is much smaller:
  - **CatBoost:** 0.6585
  - **Logistic Regression:** 0.6504

Interpretation:
These results suggest that the checkpoint features contain important **non-linear relationships** that CatBoost is better able to capture than Logistic Regression. This is especially evident at the **6-over** and **15-over** checkpoints. The results also support the supervisor feedback that training **separate models for each checkpoint** is a useful direction for Research Project B.

## Step 17: prepare Random Forest

We now add Random Forest as another checkpoint-specific baseline model.

Like Logistic Regression, scikit-learn Random Forest cannot directly use:
- text categorical columns such as `venue`
- missing values in numeric columns

So we will use the same preprocessing structure:
- numeric imputation
- categorical imputation
- one-hot encoding for `venue`
- Random Forest classifier

In [16]:
# Step 17: import Random Forest

from sklearn.ensemble import RandomForestClassifier

## Step 18: build Random Forest pipeline for checkpoint 6

We first test Random Forest on checkpoint 6 only.

Preprocessing:
- numeric features: median imputation
- categorical feature (`venue`): most-frequent imputation + one-hot encoding

Unlike Logistic Regression, Random Forest does not need feature scaling, so we keep the numeric preprocessing simpler here.

In [17]:
# Step 18: define columns and build Random Forest pipeline for checkpoint 6

# Same modelling features used before
model_features_6 = [
    "wickets",
    "run_rate",
    "momentum_runs",
    "momentum_wickets",
    "venue",
    "resource_frac",
    "venue_mean_runs_cp",
    "venue_mean_wkts_cp",
    "season_mean_runs_cp",
    "season_mean_wkts_cp",
    "elo_diff",
]

categorical_features_6 = ["venue"]
numeric_features_6 = [col for col in model_features_6 if col not in categorical_features_6]

# Recreate checkpoint 6 X and y
X_6 = df_6[model_features_6].copy()
y_6 = df_6[target_col].copy()

X6_train, X6_test, y6_train, y6_test = train_test_split(
    X_6,
    y_6,
    test_size=0.2,
    random_state=42,
    stratify=y_6
)

numeric_transformer_rf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer_rf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_rf_6 = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_rf, numeric_features_6),
        ("cat", categorical_transformer_rf, categorical_features_6),
    ]
)

rf_pipeline_6 = Pipeline(steps=[
    ("preprocessor", preprocessor_rf_6),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    ))
])

print("Numeric features:")
print(numeric_features_6)

print("\nCategorical features:")
print(categorical_features_6)

print("\nX6_train shape:", X6_train.shape)
print("X6_test shape:", X6_test.shape)
print("y6_train shape:", y6_train.shape)
print("y6_test shape:", y6_test.shape)

Numeric features:
['wickets', 'run_rate', 'momentum_runs', 'momentum_wickets', 'resource_frac', 'venue_mean_runs_cp', 'venue_mean_wkts_cp', 'season_mean_runs_cp', 'season_mean_wkts_cp', 'elo_diff']

Categorical features:
['venue']

X6_train shape: (494, 11)
X6_test shape: (124, 11)
y6_train shape: (494,)
y6_test shape: (124,)


## Step 19: train and evaluate Random Forest for checkpoint 6

In this step, we fit the Random Forest pipeline on checkpoint 6 and evaluate it using:
- accuracy
- confusion matrix
- classification report

This gives us the first direct comparison against both CatBoost and Logistic Regression at checkpoint 6.

In [18]:
# Step 19: train and evaluate Random Forest for checkpoint 6

rf_pipeline_6.fit(X6_train, y6_train)

y6_pred_rf = rf_pipeline_6.predict(X6_test)

acc_rf_6 = accuracy_score(y6_test, y6_pred_rf)
cm_rf_6 = confusion_matrix(y6_test, y6_pred_rf)
tn, fp, fn, tp = cm_rf_6.ravel()

print("Checkpoint 6 Random Forest Accuracy:")
print(acc_rf_6)

print("\nConfusion Matrix [[TN, FP], [FN, TP]]:")
print(cm_rf_6)

print("\nConfusion Matrix Breakdown:")
print(f"TN (actual 0 predicted 0): {tn}")
print(f"FP (actual 0 predicted 1): {fp}")
print(f"FN (actual 1 predicted 0): {fn}")
print(f"TP (actual 1 predicted 1): {tp}")

print("\nClassification Report:")
print(classification_report(y6_test, y6_pred_rf))

Checkpoint 6 Random Forest Accuracy:
0.5806451612903226

Confusion Matrix [[TN, FP], [FN, TP]]:
[[41 25]
 [27 31]]

Confusion Matrix Breakdown:
TN (actual 0 predicted 0): 41
FP (actual 0 predicted 1): 25
FN (actual 1 predicted 0): 27
TP (actual 1 predicted 1): 31

Classification Report:
              precision    recall  f1-score   support

           0       0.60      0.62      0.61        66
           1       0.55      0.53      0.54        58

    accuracy                           0.58       124
   macro avg       0.58      0.58      0.58       124
weighted avg       0.58      0.58      0.58       124



## Step 20: compare CatBoost, Logistic Regression, and Random Forest at checkpoint 6

This step creates one small comparison table for checkpoint 6 across the three models tested so far:
- CatBoost
- Logistic Regression
- Random Forest

This helps decide which models are worth extending to all four checkpoints.

In [19]:
# Step 20: compare all three models at checkpoint 6

cat_cm_6 = confusion_matrix(y6_test, y6_pred)
cat_tn, cat_fp, cat_fn, cat_tp = cat_cm_6.ravel()

lr_cm_6 = confusion_matrix(y6_test, y6_pred_lr)
lr_tn, lr_fp, lr_fn, lr_tp = lr_cm_6.ravel()

rf_cm_6 = confusion_matrix(y6_test, y6_pred_rf)
rf_tn, rf_fp, rf_fn, rf_tp = rf_cm_6.ravel()

comparison_cp6_all = pd.DataFrame([
    {
        "checkpoint": 6,
        "model": "CatBoost",
        "accuracy": accuracy_score(y6_test, y6_pred),
        "tn": cat_tn,
        "fp": cat_fp,
        "fn": cat_fn,
        "tp": cat_tp
    },
    {
        "checkpoint": 6,
        "model": "Logistic Regression",
        "accuracy": accuracy_score(y6_test, y6_pred_lr),
        "tn": lr_tn,
        "fp": lr_fp,
        "fn": lr_fn,
        "tp": lr_tp
    },
    {
        "checkpoint": 6,
        "model": "Random Forest",
        "accuracy": accuracy_score(y6_test, y6_pred_rf),
        "tn": rf_tn,
        "fp": rf_fp,
        "fn": rf_fn,
        "tp": rf_tp
    }
])

comparison_cp6_all["accuracy"] = comparison_cp6_all["accuracy"].round(4)
comparison_cp6_all = comparison_cp6_all.sort_values("accuracy", ascending=False).reset_index(drop=True)

print("Checkpoint 6 comparison across models:")
display(comparison_cp6_all)

Checkpoint 6 comparison across models:


,checkpoint,model,accuracy,tn,fp,fn,tp
0,6,CatBoost,0.7097,47,19,17,41
1,6,Logistic Regression,0.6452,46,20,24,34
2,6,Random Forest,0.5806,41,25,27,31


## Step 21: run Random Forest separately for all checkpoints

We now repeat the same checkpoint-specific workflow for Random Forest across:
- 6 overs
- 10 overs
- 15 overs
- 20 overs

This helps confirm whether the weak checkpoint 6 result is consistent across all checkpoints or only specific to the early innings stage.

In [20]:
# Step 21: Random Forest loop across all checkpoints with detailed evaluation

feature_cols_model = [
    "wickets",
    "run_rate",
    "momentum_runs",
    "momentum_wickets",
    "venue",
    "resource_frac",
    "venue_mean_runs_cp",
    "venue_mean_wkts_cp",
    "season_mean_runs_cp",
    "season_mean_wkts_cp",
    "elo_diff",
]

categorical_features = ["venue"]
numeric_features = [col for col in feature_cols_model if col not in categorical_features]
checkpoints = [6, 10, 15, 20]

rf_detailed_results = []
rf_classification_reports = {}

for cp in checkpoints:
    d = model_df[model_df["checkpoint"] == cp].copy()

    X_cp = d[feature_cols_model].copy()
    y_cp = d[target_col].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X_cp,
        y_cp,
        test_size=0.2,
        random_state=42,
        stratify=y_cp
    )

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

    rf_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            random_state=42,
            n_jobs=-1
        ))
    ])

    rf_pipeline.fit(X_train, y_train)
    y_pred = rf_pipeline.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    report_dict = classification_report(y_test, y_pred, output_dict=True)
    report_df = pd.DataFrame(report_dict).transpose()
    rf_classification_reports[cp] = report_df

    rf_detailed_results.append({
        "checkpoint": cp,
        "n_rows": len(d),
        "n_train": len(X_train),
        "n_test": len(X_test),
        "accuracy": acc,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    })

    print(f"\n{'='*70}")
    print(f"RANDOM FOREST | CHECKPOINT {cp}")
    print(f"{'='*70}")
    print(f"Accuracy: {acc:.4f}")

    print("\nConfusion Matrix [[TN, FP], [FN, TP]]:")
    print(cm)

    print("\nConfusion Matrix Breakdown:")
    print(f"TN (actual 0 predicted 0): {tn}")
    print(f"FP (actual 0 predicted 1): {fp}")
    print(f"FN (actual 1 predicted 0): {fn}")
    print(f"TP (actual 1 predicted 1): {tp}")

rf_detailed_results_df = pd.DataFrame(rf_detailed_results).sort_values("checkpoint").reset_index(drop=True)

print("\nOverall Random Forest detailed results summary:")
display(rf_detailed_results_df)


RANDOM FOREST | CHECKPOINT 6
Accuracy: 0.5806

Confusion Matrix [[TN, FP], [FN, TP]]:
[[41 25]
 [27 31]]

Confusion Matrix Breakdown:
TN (actual 0 predicted 0): 41
FP (actual 0 predicted 1): 25
FN (actual 1 predicted 0): 27
TP (actual 1 predicted 1): 31

RANDOM FOREST | CHECKPOINT 10
Accuracy: 0.5854

Confusion Matrix [[TN, FP], [FN, TP]]:
[[42 23]
 [28 30]]

Confusion Matrix Breakdown:
TN (actual 0 predicted 0): 42
FP (actual 0 predicted 1): 23
FN (actual 1 predicted 0): 28
TP (actual 1 predicted 1): 30

RANDOM FOREST | CHECKPOINT 15
Accuracy: 0.6364

Confusion Matrix [[TN, FP], [FN, TP]]:
[[38 25]
 [19 39]]

Confusion Matrix Breakdown:
TN (actual 0 predicted 0): 38
FP (actual 0 predicted 1): 25
FN (actual 1 predicted 0): 19
TP (actual 1 predicted 1): 39

RANDOM FOREST | CHECKPOINT 20
Accuracy: 0.6174

Confusion Matrix [[TN, FP], [FN, TP]]:
[[39 20]
 [24 32]]

Confusion Matrix Breakdown:
TN (actual 0 predicted 0): 39
FP (actual 0 predicted 1): 20
FN (actual 1 predicted 0): 24
TP (act

,checkpoint,n_rows,n_train,n_test,accuracy,tn,fp,fn,tp
0,6,618,494,124,0.580645,41,25,27,31
1,10,613,490,123,0.585366,42,23,28,30
2,15,601,480,121,0.636364,38,25,19,39
3,20,572,457,115,0.617391,39,20,24,32


## Step 22: compare CatBoost, Logistic Regression, and Random Forest across all checkpoints

We now combine the checkpoint-specific results from all three models into a single comparison table.

This makes it easier to:
- compare accuracy across checkpoints
- identify the strongest model overall
- identify whether any model performs better at a specific innings stage

In [21]:
# Step 22: compare all three models across all checkpoints

catboost_compare = detailed_results_df.copy()
catboost_compare["model"] = "CatBoost"

logreg_compare = lr_detailed_results_df.copy()
logreg_compare["model"] = "Logistic Regression"

rf_compare = rf_detailed_results_df.copy()
rf_compare["model"] = "Random Forest"

three_model_comparison = pd.concat(
    [catboost_compare, logreg_compare, rf_compare],
    ignore_index=True
)

three_model_comparison = three_model_comparison[
    ["checkpoint", "model", "accuracy", "tn", "fp", "fn", "tp"]
].sort_values(["checkpoint", "accuracy"], ascending=[True, False]).reset_index(drop=True)

three_model_comparison["accuracy"] = three_model_comparison["accuracy"].round(4)

print("Three-model comparison across all checkpoints:")
display(three_model_comparison)

Three-model comparison across all checkpoints:


,checkpoint,model,accuracy,tn,fp,fn,tp
0,6,CatBoost,0.7097,47,19,17,41
1,6,Logistic Regression,0.6452,46,20,24,34
2,6,Random Forest,0.5806,41,25,27,31
3,10,CatBoost,0.6585,50,15,27,31
4,10,Logistic Regression,0.6504,45,20,23,35
5,10,Random Forest,0.5854,42,23,28,30
6,15,CatBoost,0.7190,39,24,10,48
7,15,Logistic Regression,0.6529,40,23,19,39
8,15,Random Forest,0.6364,38,25,19,39
9,20,CatBoost,0.6957,41,18,17,39


In [22]:
# Optional: pivot table for quick accuracy comparison

accuracy_pivot_3models = (
    three_model_comparison
    .pivot(index="checkpoint", columns="model", values="accuracy")
    .reset_index()
)

print("Accuracy comparison by checkpoint:")
display(accuracy_pivot_3models)

Accuracy comparison by checkpoint:


model,checkpoint,CatBoost,Logistic Regression,Random Forest
0,6,0.7097,0.6452,0.5806
1,10,0.6585,0.6504,0.5854
2,15,0.7190,0.6529,0.6364
3,20,0.6957,0.6696,0.6174


## Step 23: interpretation of the three-model checkpoint comparison

The checkpoint-specific comparison across three models shows a clear and consistent ranking:

- **CatBoost** performs best at every checkpoint
- **Logistic Regression** performs second at every checkpoint
- **Random Forest** performs worst at every checkpoint

Key observations:
- The best overall result is achieved by **CatBoost at 15 overs**:
  - **Accuracy:** 0.7190
- CatBoost also performs strongly at:
  - **6 overs:** 0.7097
  - **20 overs:** 0.6957
- Logistic Regression is more competitive at **10 overs** and **20 overs**, but still remains below CatBoost
- Random Forest underperforms consistently across all checkpoints, suggesting that the current feature set and model configuration are not capturing the checkpoint structure as effectively as CatBoost

Interpretation:
These results suggest that the prediction problem benefits from a model that can capture non-linear interactions and complex threshold effects in the checkpoint-based match-state features. Among the models tested so far, **CatBoost is the strongest candidate** for the main Research Project B checkpoint-specific modelling approach.

In [23]:
# Step 24: check XGBoost availability

from xgboost import XGBClassifier

print("XGBoost imported successfully.")

XGBoost imported successfully.


## Step 25: run XGBoost separately for all checkpoints

We now evaluate XGBoost across all four checkpoints using the same checkpoint-specific workflow.

Preprocessing:
- numeric features: median imputation
- categorical feature (`venue`): most-frequent imputation + one-hot encoding

This allows direct comparison with:
- CatBoost
- Logistic Regression
- Random Forest

In [24]:
# Step 25: XGBoost loop across all checkpoints with detailed evaluation

from xgboost import XGBClassifier

feature_cols_model = [
    "wickets",
    "run_rate",
    "momentum_runs",
    "momentum_wickets",
    "venue",
    "resource_frac",
    "venue_mean_runs_cp",
    "venue_mean_wkts_cp",
    "season_mean_runs_cp",
    "season_mean_wkts_cp",
    "elo_diff",
]

categorical_features = ["venue"]
numeric_features = [col for col in feature_cols_model if col not in categorical_features]
checkpoints = [6, 10, 15, 20]

xgb_detailed_results = []
xgb_classification_reports = {}

for cp in checkpoints:
    d = model_df[model_df["checkpoint"] == cp].copy()

    X_cp = d[feature_cols_model].copy()
    y_cp = d[target_col].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X_cp,
        y_cp,
        test_size=0.2,
        random_state=42,
        stratify=y_cp
    )

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

    xgb_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42
        ))
    ])

    xgb_pipeline.fit(X_train, y_train)
    y_pred = xgb_pipeline.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    report_dict = classification_report(y_test, y_pred, output_dict=True)
    report_df = pd.DataFrame(report_dict).transpose()
    xgb_classification_reports[cp] = report_df

    xgb_detailed_results.append({
        "checkpoint": cp,
        "n_rows": len(d),
        "n_train": len(X_train),
        "n_test": len(X_test),
        "accuracy": acc,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    })

    print(f"\n{'='*70}")
    print(f"XGBOOST | CHECKPOINT {cp}")
    print(f"{'='*70}")
    print(f"Accuracy: {acc:.4f}")

    print("\nConfusion Matrix [[TN, FP], [FN, TP]]:")
    print(cm)

    print("\nConfusion Matrix Breakdown:")
    print(f"TN (actual 0 predicted 0): {tn}")
    print(f"FP (actual 0 predicted 1): {fp}")
    print(f"FN (actual 1 predicted 0): {fn}")
    print(f"TP (actual 1 predicted 1): {tp}")

xgb_detailed_results_df = pd.DataFrame(xgb_detailed_results).sort_values("checkpoint").reset_index(drop=True)

print("\nOverall XGBoost detailed results summary:")
display(xgb_detailed_results_df)


XGBOOST | CHECKPOINT 6
Accuracy: 0.6048

Confusion Matrix [[TN, FP], [FN, TP]]:
[[42 24]
 [25 33]]

Confusion Matrix Breakdown:
TN (actual 0 predicted 0): 42
FP (actual 0 predicted 1): 24
FN (actual 1 predicted 0): 25
TP (actual 1 predicted 1): 33

XGBOOST | CHECKPOINT 10
Accuracy: 0.5935

Confusion Matrix [[TN, FP], [FN, TP]]:
[[40 25]
 [25 33]]

Confusion Matrix Breakdown:
TN (actual 0 predicted 0): 40
FP (actual 0 predicted 1): 25
FN (actual 1 predicted 0): 25
TP (actual 1 predicted 1): 33

XGBOOST | CHECKPOINT 15
Accuracy: 0.6364

Confusion Matrix [[TN, FP], [FN, TP]]:
[[35 28]
 [16 42]]

Confusion Matrix Breakdown:
TN (actual 0 predicted 0): 35
FP (actual 0 predicted 1): 28
FN (actual 1 predicted 0): 16
TP (actual 1 predicted 1): 42

XGBOOST | CHECKPOINT 20
Accuracy: 0.6609

Confusion Matrix [[TN, FP], [FN, TP]]:
[[44 15]
 [24 32]]

Confusion Matrix Breakdown:
TN (actual 0 predicted 0): 44
FP (actual 0 predicted 1): 15
FN (actual 1 predicted 0): 24
TP (actual 1 predicted 1): 32



,checkpoint,n_rows,n_train,n_test,accuracy,tn,fp,fn,tp
0,6,618,494,124,0.604839,42,24,25,33
1,10,613,490,123,0.593496,40,25,25,33
2,15,601,480,121,0.636364,35,28,16,42
3,20,572,457,115,0.660870,44,15,24,32


## Step 26: compare CatBoost, Logistic Regression, Random Forest, and XGBoost across all checkpoints

We now combine the checkpoint-specific results from all four models into a single comparison table.

This helps identify:
- the strongest model overall
- the strongest model at each checkpoint
- whether additional models outperform CatBoost

In [25]:
# Step 26: compare all four models across all checkpoints

catboost_compare = detailed_results_df.copy()
catboost_compare["model"] = "CatBoost"

logreg_compare = lr_detailed_results_df.copy()
logreg_compare["model"] = "Logistic Regression"

rf_compare = rf_detailed_results_df.copy()
rf_compare["model"] = "Random Forest"

xgb_compare = xgb_detailed_results_df.copy()
xgb_compare["model"] = "XGBoost"

four_model_comparison = pd.concat(
    [catboost_compare, logreg_compare, rf_compare, xgb_compare],
    ignore_index=True
)

four_model_comparison = four_model_comparison[
    ["checkpoint", "model", "accuracy", "tn", "fp", "fn", "tp"]
].sort_values(["checkpoint", "accuracy"], ascending=[True, False]).reset_index(drop=True)

four_model_comparison["accuracy"] = four_model_comparison["accuracy"].round(4)

print("Four-model comparison across all checkpoints:")
display(four_model_comparison)

Four-model comparison across all checkpoints:


,checkpoint,model,accuracy,tn,fp,fn,tp
0,6,CatBoost,0.7097,47,19,17,41
1,6,Logistic Regression,0.6452,46,20,24,34
2,6,XGBoost,0.6048,42,24,25,33
3,6,Random Forest,0.5806,41,25,27,31
4,10,CatBoost,0.6585,50,15,27,31
5,10,Logistic Regression,0.6504,45,20,23,35
6,10,XGBoost,0.5935,40,25,25,33
7,10,Random Forest,0.5854,42,23,28,30
8,15,CatBoost,0.7190,39,24,10,48
9,15,Logistic Regression,0.6529,40,23,19,39


In [26]:
# Optional: pivot table for quick accuracy comparison

accuracy_pivot_4models = (
    four_model_comparison
    .pivot(index="checkpoint", columns="model", values="accuracy")
    .reset_index()
)

print("Accuracy comparison by checkpoint:")
display(accuracy_pivot_4models)

Accuracy comparison by checkpoint:


model,checkpoint,CatBoost,Logistic Regression,Random Forest,XGBoost
0,6,0.7097,0.6452,0.5806,0.6048
1,10,0.6585,0.6504,0.5854,0.5935
2,15,0.7190,0.6529,0.6364,0.6364
3,20,0.6957,0.6696,0.6174,0.6609


## Step 27: interpretation of the four-model checkpoint comparison

The four-model comparison shows a consistent ranking across all checkpoints:

- **CatBoost** performs best at every checkpoint
- **Logistic Regression** performs second at every checkpoint
- **XGBoost**, with the current configuration, does not outperform CatBoost or Logistic Regression
- **Random Forest** performs worst overall

Key observations:
- The best overall result remains **CatBoost at 15 overs**
  - **Accuracy:** 0.7190
- CatBoost is also strong at:
  - **6 overs:** 0.7097
  - **20 overs:** 0.6957
- Logistic Regression remains the strongest non-boosting baseline in this experiment
- XGBoost does not currently provide an improvement over CatBoost and is also below Logistic Regression at all checkpoints
- Random Forest underperforms consistently, suggesting that it is less suitable for this checkpoint-based prediction setting under the current feature configuration

Conclusion:
At this stage, **CatBoost is the strongest candidate for the main Research Project B checkpoint-specific modelling approach**. The next logical step is to tune CatBoost, starting with the best-performing checkpoint (**15 overs**), before considering additional models such as SVM.

In [27]:
# Step 28: prepare checkpoint 15 data for CatBoost tuning

df_15 = model_df[model_df["checkpoint"] == 15].copy()

feature_cols_model = [
    "wickets",
    "run_rate",
    "momentum_runs",
    "momentum_wickets",
    "venue",
    "resource_frac",
    "venue_mean_runs_cp",
    "venue_mean_wkts_cp",
    "season_mean_runs_cp",
    "season_mean_wkts_cp",
    "elo_diff",
]

categorical_features = ["venue"]

X_15 = df_15[feature_cols_model].copy()
y_15 = df_15[target_col].copy()

X15_train, X15_test, y15_train, y15_test = train_test_split(
    X_15,
    y_15,
    test_size=0.2,
    random_state=42,
    stratify=y_15
)

print("Checkpoint 15 shape:", df_15.shape)
print("Train shape:", X15_train.shape, y15_train.shape)
print("Test shape:", X15_test.shape, y15_test.shape)

print("\nCheckpoint 15 target distribution:")
print(y_15.value_counts().sort_index())

print("\nMissing values in X15_train:")
print(X15_train.isna().sum()[X15_train.isna().sum() > 0])

Checkpoint 15 shape: (601, 14)
Train shape: (480, 11) (480,)
Test shape: (121, 11) (121,)

Checkpoint 15 target distribution:
target_batfirst
0    315
1    286
Name: count, dtype: int64

Missing values in X15_train:
elo_diff    6
dtype: int64
